In [ ]:
# =============================================================================
# FUMD-AI Preprocessing Workflow -- Step 3: Generate the OMNeT++ feature matrix
# =============================================================================
# Step:         3 of 7 (input stage for the core labeling chain)
# Summary:      Clean raw OMNeT++ vector export and pivot into a wide per-vehicle-per-timestep feature matrix.
#
# Author(s):
#   - Cristina Bernad (ORCID: 0000-0001-9537-415X)
#   - Sonja Filiposka <sonja.filiposka@finki.ukim.mk> (ORCID: 0000-0003-0034-2855)
#   - Katja Gilly (ORCID: 0000-0002-8985-0639)
#
# Copyright:    (c) 2026 Cristina Bernad, Sonja Filiposka, Katja Gilly
# Repository:   https://github.com/FUMD-AI/fumd-ai-preprocessing-workflow
# Version:      1.0.2
# Funding:      This work has been funded by the FUMD-AI project, an EOSC GRAVITY -
#             Inter Project with Grant Number 25-EOSC-GRV-INTER-013.
#
# -----------------------------------------------------------------------------
# Licence
# Unless otherwise indicated:
#
#   * Source code in this notebook is licensed under the MIT License.
#
#   * Explanatory text and original figures are licensed under Creative
#     Commons Attribution 4.0 International (CC BY 4.0). Input datasets
#     retain the licences stated in their corresponding metadata or
#     source records.
#
# SPDX-License-Identifier: MIT
# -----------------------------------------------------------------------------
#
# Structured, machine-readable metadata for this workflow (authors, license,
# inputs/outputs per step) is also maintained in ro-crate-metadata.json at
# the repository root - update both together if either changes.
# =============================================================================


# Step 3 — Generate the OMNeT++ feature matrix

Part of the **FUMD-AI preprocessing workflow**: turns raw SUMO + OMNeT++
simulation output into a labeled, AI-ready dataset of cellular handover
events. This is step 3 of 7 (step 7 is optional/exploratory).

**Purpose.** OMNeT++ exports one row per `(time, module, metric)` observation.
This notebook cleans that raw export and pivots it into a wide table with one
row per `(vehicle, time)` and one column per network-quality metric (SINR,
CQI, RLC delay/throughput, serving cell, distance to the serving gNB, ...).
It also adds lagged and lead `servingCell` columns
(`servingCell-1..-7` / `servingCell1..7`) capturing which cell each vehicle
was connected to N seconds in the past/future.

**Input:** an OMNeT++ vector export - either a raw `extractvectors` CSV or
the compact 4-column TSV this project's `extract_vec_pure_python.py`
produces directly from a `.vec` file. `extractvectors` builds vary in
exactly what they emit (see fix 7 below), but the cleaning step in this
notebook normalizes all of the variants seen on this project down to the
same shape: `Time  Object  Vector  Value`, e.g.:

```
Time    Object                                  Vector                  Value
0.1     car[0].cellularNic.phy                  "servingCell """""      0.0
0.1     car[0].cellularNic.nrChannelModel[0]    "distance """""         730.321606
0.104   car[0].cellularNic.nrPhy                "averageCqiDl """""     10.0
0.107   car[0].cellularNic.nrRlc.um             "rlcThroughputDl """""  644.859813
```

**Output:** `<OUTPUT_MATRIX_PATH>` — one row per vehicle per resampled tick.

**Note:** this notebook never modifies the raw input file in place. All
intermediate/cleaned files are written fresh on every run, so it is always
safe to re-run from scratch.

**System requirement:** a `sed` binary on `PATH` (used for fast text
cleaning of the raw export, which can be tens of GB — a pure-pandas
string-replace pass over that many rows would be far slower and far more
memory-hungry). The cleaning pattern is written portably (literal tab
bytes rather than a `	` escape) so it works identically with both GNU
`sed` (default on Linux) and the BSD `sed` shipped by default on macOS -
no need to install GNU sed separately on Mac.


In [ ]:
import subprocess
import pandas as pd


In [ ]:
# ---- Parameters (edit for your own simulation run) ----
# This cell is tagged "parameters" so the notebook can also be executed
# headlessly with papermill, e.g.:
#   papermill step_3_generate_omnet_matrix.ipynb out.ipynb -p RAW_OMNET_PATH my_run_omnet_export.csv

# Default points at the tiny bundled example (example-data/) so this
# notebook runs out of the box without needing Step 1's extractvectors
# step - replace with your own Step 1 output (or raw export) directly.
RAW_OMNET_PATH = "example-data/raw_omnet_export.csv"  # raw OMNeT++ vector export (input)
CLEAN_OMNET_PATH = "omnet_clean.csv"                  # intermediate cleaned/parsed file (generated)
OUTPUT_MATRIX_PATH = "omnet_feature_matrix.csv"        # final wide-format matrix (output) -> feeds Step 4's OMNET_PATH

RESAMPLE_FREQ = "10ms"                    # uniform time grid to resample onto
LAG_LEAD_SECONDS = [1, 2, 3, 4, 5, 6, 7]  # adds servingCell-N / servingCellN columns

# Metrics that never carry a nonzero value anywhere in this simulation setup
# (either zero recorded rows at all, or every recorded row is exactly 0) and
# add no information; dropped right after pivoting. Checked directly against
# the full VoipDl-Urban-900_1 run: rlcPacketLossTotal/rlcPduPacketLossDl have
# zero recorded rows for any vehicle; rlcPduThroughputDl has ~880k recorded
# rows (so it looks populated at a glance) but every single one is exactly
# 0.000000, including for vehicles whose near-identical rlcThroughputDl
# metric shows plenty of real nonzero values at the same timestamps - so
# it's genuinely uninformative here, not just sparsely recorded. Remove
# entries here if your OMNeT++ config actually records nonzero values for
# them.
ALWAYS_ZERO_VECTORS = ["rlcPacketLossTotal ", "rlcPduPacketLossDl ", "rlcPduThroughputDl "]

# RLC downlink metrics that are only ever recorded for vehicles with actual
# downlink application traffic (e.g. an assigned VoIP flow) - vehicles with
# none never get these vectors declared at all, not even one zero-valued
# row, so per-vehicle ffill/bfill has nothing to work with for them. Checked
# directly against VoipDl-Urban-900_1: these metrics are missing together
# for roughly half of all vehicles, while the other half have plenty of
# real, nonzero values - this is a genuine "no downlink traffic for this
# vehicle" case, not a data quality problem, so it would be wrong to drop
# the columns outright the way ALWAYS_ZERO_VECTORS does above. Any of these
# still NaN after fill are filled with 0 ("no downlink RLC activity observed
# for this vehicle") instead. Remove entries here if your OMNeT++ config
# records these for every vehicle regardless of traffic (in which case a
# genuinely-missing value would indicate a real bug worth investigating, and
# the fill would silently hide it).
NO_TRAFFIC_FILL_ZERO_VECTORS = ["rlcDelayDl ", "rlcPduDelayDl ", "rlcThroughputDl "]

MAX_ROWS = None  # set an integer to only read the first N raw rows (quick test runs)


## 1. Clean the raw OMNeT++ export

The raw export needs several fixes before it can be read as a normal wide CSV:

1. Normalize whatever a given `extractvectors` build put in the `Vector`
   field down to a bare metric name. Some builds emit just the name
   (`servingCell`); others bundle the full declaration string plus an
   (almost always empty) `Split` field into the same tab-delimited column
   (`"servingCell:vector ETV" ""`). Both are reduced to `servingCell `.
2. Drop a redundant `servingCell` line that OMNeT++ logs under the wrong
   module (`cellularNic.phy` instead of `cellularNic.nrPhy`) - regardless
   of whether the module path also carries a network-instance prefix
   (e.g. `NRSeveralBSALC.car[3].cellularNic.phy`).
3. Collapse `<prefix>car[i].cellularNic.<submodule>` object names down to
   just `<prefix>i` (the submodule is implied by the metric name already).
4. Drop `receivedPacketFromLowerLayer` rows (not one of the metrics we keep).
5. Strip the `car[...]` wrapper down to a plain integer vehicle id.
6. Strip stray double quotes left by OMNeT++'s vector export format.
7. Some `extractvectors` builds also prepend an event number and an
   internal numeric object id before `Time`/`Object`
   (`<event>\t<time>\t<objectId>\t<object>...`), and print a
   space-separated (not tab-separated) header line. Both are handled by
   discarding whatever header the raw file has and writing a fixed one,
   and by stripping the leading `<event>\t...\t<objectId>\t` prefix
   whenever it's present (a no-op otherwise).

All fixes are applied in a single `sed` pass for speed on multi-GB
files, streaming straight from `RAW_OMNET_PATH` into a **new**
`CLEAN_OMNET_PATH` file — the raw input is never touched.

**Why dropping the `cellularNic.phy` `servingCell` row (fix 2) is enough.**
Some simu5g network configurations (e.g. `NRSeveralBSALC`, dual-connectivity
capable) declare *both* an LTE-style stack (`cellularNic.rlc.um`,
`cellularNic.phy`) and the 5G NR stack (`cellularNic.nrRlc.um`,
`cellularNic.nrPhy`) per vehicle, under the *same* metric names. If both
stacks actually carried data, collapsing the module path down to just the
metric name (fix 3) would silently conflate two different signals under
one column. Checked directly against real raw `.vec` and `extractvectors`
CSV output from this project (`VoipDl-Urban-900_1`, multiple vehicles):
the LTE-style stack's `servingCell` is recorded but is constant `0` for
the entire simulation (it is never actually connected to a base station
in this scenario), and every *other* LTE-style metric (`rlcDelayDl`,
`averageCqiDl`, etc.) has **zero** recorded data rows at all - only the NR
stack ever carries real values. So dropping just the one row fix 2 targets
is sufficient here; if you use a network configuration where the
non-NR stack is genuinely populated with data, this cleaning step will
need to explicitly filter to the NR module path instead.

**Note on fix 2's module-path prefix.** An earlier version of this rule
only matched `car[N].cellularNic.phy` when that appeared immediately after
a tab, with nothing in front of it. That's only true when the OMNeT++
network's top-level name is stripped from the module path. For a network
like `NRSeveralBSALC`, the module path keeps that prefix
(`NRSeveralBSALC.car[3].cellularNic.phy`), so the old rule silently never
matched and both the constant-0 and real `servingCell` values survived
into the pivot under the same column - `.aggregate("min")` then collapsed
them to `0` wherever both existed at the same timestamp, corrupting the
whole `servingCell` signal. The rule now allows an arbitrary non-tab
prefix before `car[`, so it matches regardless of the network name.


In [ ]:
def clean_omnet_raw(raw_path: str, clean_path: str) -> None:
    """Stream-clean a raw OMNeT++ vector export into a pandas-readable TSV.

    Non-destructive: never modifies `raw_path`. Fully overwrites
    `clean_path` on every call, so re-running this cell is always safe.
    """
    # Tab characters are embedded here as literal bytes (via TAB), not as a
    # "\t" escape sequence, so this sed script behaves identically under
    # GNU sed (Linux) and the BSD sed shipped by default on macOS. BSD sed
    # does not treat "\t" in a pattern as a tab character - it matches the
    # literal letter t - so relying on that escape silently mis-cleans the
    # file on macOS (no error, just corrupted output) instead of failing
    # loudly. A literal tab byte in the pattern is unambiguous on both.
    TAB = "\t"
    sed_script = (
        # fix 7 (part 1): drop a leading "<event>\t<time>\t<objectId>\t"
        # prefix some extractvectors builds add before Time/Object - a no-op
        # if the line already starts with a plain "<time>\t" (Time itself
        # is never a bare integer immediately followed by a tab, since it's
        # always written with a decimal point).
        r's/^[0-9]+' + TAB + r'([0-9]+\.[0-9]+)' + TAB + r'[0-9]+' + TAB + r'/\1' + TAB + r'/; '
        # fix 1: collapse a bundled "<name>:vector<...> <FORMAT>" declaration
        # string (with an empty Split field glued on after a space) down to
        # "<name> " - a no-op if the Vector field is already a bare name.
        r's/:vector[^' + TAB + r']*' + TAB + r'/ ' + TAB + r'/; '
        # fix 2: drop the redundant LTE-stack servingCell row (see markdown
        # above). "[^TAB]*" before "car[" tolerates any network-name prefix;
        # the optional trailing quote handles the Object field still being
        # quoted at this point in the pipeline (quotes are stripped in fix 6).
        r'/' + TAB + r'[^' + TAB + r']*car\[[0-9]+\]\.cellularNic\.phy"?' + TAB + r'"*servingCell/d; '
        # fix 3: collapse "<prefix>car[i].cellularNic.<submodule>" down to "<prefix>car[i]".
        r's/\]\.cellularNic\.[^' + TAB + r']*' + TAB + r'/' + TAB + r'/; '
        # fix 4: drop receivedPacketFromLowerLayer rows (not a metric we keep).
        r'/receivedPacketFromLowerLayer/d; '
        # fix 5: strip the "car[...]" wrapper down to a plain integer vehicle id.
        r's/car\[//g; '
        # fix 6: strip stray double quotes left by OMNeT++'s vector export format.
        r's/"//g'
    )
    with open(clean_path, "w") as clean_f:
        # fix 7 (part 2): don't trust the raw file's own header line - some
        # extractvectors builds print it space-separated instead of
        # tab-separated, which pandas would otherwise misread entirely.
        # Discard it and write the canonical tab-separated header ourselves.
        #
        # `tail -n +2` (not a Python file iterator) is what skips that raw
        # header line here, deliberately. A Python text file object does
        # internal read-ahead buffering: calling next()/readline() on it to
        # skip one line can silently advance the underlying OS file
        # descriptor far past that line, and a subprocess reading from that
        # same fd afterwards (as sed does here, for speed on multi-GB files)
        # starts from wherever the OS position actually ended up - losing a
        # chunk of data and starting sed's input mid-line. Piping through a
        # separate `tail` process avoids this entirely, since only `tail`
        # ever touches the raw file's own fd.
        clean_f.write(f"Time{TAB}Object{TAB}Vector{TAB}Value\n")
        clean_f.flush()
        tail_proc = subprocess.Popen(["tail", "-n", "+2", raw_path], stdout=subprocess.PIPE)
        subprocess.run(["sed", "-E", sed_script], stdin=tail_proc.stdout, stdout=clean_f, check=True)
        tail_proc.stdout.close()
        tail_proc.wait()


clean_omnet_raw(RAW_OMNET_PATH, CLEAN_OMNET_PATH)
print("cleaned file written to", CLEAN_OMNET_PATH)


## 2. Load, pivot to wide format, and fill small per-vehicle gaps

In [ ]:
# Load the cleaned export. It is still in "long" format at this point:
# one row per (Time, Object=vehicle id, Vector=metric name, Value).
omnet = pd.read_csv(CLEAN_OMNET_PATH, delimiter="\t", nrows=MAX_ROWS)

# Object may still carry the OMNeT++ network's own top-level module name as
# a literal prefix (e.g. "NRSeveralBSALC.5") - the sed cleaning above only
# ever collapses everything from "car[i]" onward down to the bare vehicle
# id, since it has no way to know the network\'s own name (that varies per
# project, and isn\'t part of any of the patterns it matches). Left as a
# string, this merges incorrectly (or fails outright with a dtype error)
# against the SUMO<->OMNeT id mapping used in Step 4, which is bare
# integers - so reduce Object to just its trailing integer id here,
# regardless of whatever prefix precedes it.
omnet["Object"] = omnet["Object"].astype(str).str.extract(r"(\d+)$")[0].astype(int)

print("raw rows:", len(omnet))
omnet.head()


In [ ]:
# long -> wide: pivot "Vector" (metric name) out into its own column, so each
# (Time, Object) pair becomes one row with one column per metric. `.aggregate("min")`
# is only there to collapse the rare case of duplicate (Time, Object, Vector)
# rows into a single value - it does not change anything for the normal case
# of one observation per metric per timestep.
wide = omnet.groupby(["Time", "Object", "Vector"])["Value"].aggregate("min").unstack()
wide = wide.reset_index().set_index(["Object", "Time"])

# Drop metrics known to never carry a nonzero value anywhere in this
# simulation setup (see ALWAYS_ZERO_VECTORS above) now, before the
# ffill/bfill and NaN check below - not in the resample step further down
# as an earlier version of this notebook did. That ordering was a real bug:
# these columns can be entirely absent for some vehicles just like the
# NO_TRAFFIC_FILL_ZERO_VECTORS ones, which the NaN assertion below would
# catch and fail on even though the column was about to be dropped anyway.
wide = wide.drop(columns=[c for c in ALWAYS_ZERO_VECTORS if c in wide.columns])

# Not every metric is logged at every timestep (OMNeT++ only logs a value
# when it changes). Forward-fill first (carry the last known value forward),
# then back-fill (so the very first rows of a vehicle's trip, before its
# first observation, get its first known value instead of staying empty).
wide = wide.groupby("Object").ffill()
wide = wide.groupby("Object").bfill()

# Vehicles with no downlink application traffic at all never get the RLC
# downlink metrics in NO_TRAFFIC_FILL_ZERO_VECTORS recorded even once, so
# ffill/bfill has nothing to carry forward/backward for them - see that
# parameter's definition above. Fill those specific columns' remaining gaps
# with 0 ("no downlink RLC activity observed"); every other column is left
# alone, so the assertion below still catches genuinely unexpected gaps.
for _col in NO_TRAFFIC_FILL_ZERO_VECTORS:
    if _col in wide.columns:
        wide[_col] = wide[_col].fillna(0)

wide = wide.reset_index().sort_values(by=["Time", "Object"])

assert not wide.isnull().values.any(), "unexpected NaNs remain after fill"
wide.head()


## 3. Resample onto a uniform time grid per vehicle

In [ ]:
# pandas .resample() needs a proper datetime/timedelta index, not raw floats
# (ALWAYS_ZERO_VECTORS columns are already dropped, back in step 2, above)
wide["Time"] = pd.to_timedelta(wide["Time"], unit="s")
wide = wide.set_index("Time")

print("servingCell values before resampling:", sorted(wide["servingCell "].dropna().unique()))


In [ ]:
# Resample each vehicle independently onto the uniform RESAMPLE_FREQ grid.
# Every metric is averaged over each resample window, except servingCell,
# which takes the *last* value in the window so it always stays a whole
# cell id (never a meaningless decimal average of two different cells,
# e.g. cell 3 and cell 4 averaging to 3.5).
#
# Grouping by ["Object", pd.Grouper(freq=...)] rather than doing
# .groupby("Object").resample(...) is deliberate: the groupby+resample form
# routes through DataFrameGroupBy.apply() internally, whose handling of the
# grouping columns changed across pandas versions (an `include_groups`
# option was added in pandas 2.2 to control it, but passing that same
# keyword straight into .resample() raises a TypeError on pandas < 2.2,
# since .resample() itself never accepted it - it only ever belonged to
# .apply()). Grouping with pd.Grouper avoids that version-dependent code
# path entirely and behaves identically on pandas 2.1 and 2.2+.
grouped = wide.groupby(["Object", pd.Grouper(freq=RESAMPLE_FREQ)])
final = grouped.mean(numeric_only=True)
final["servingCell "] = grouped["servingCell "].last().reindex(final.index).astype("Int64")

# resampling can introduce empty ticks at the very edges of a vehicle's
# trip (before its first / after its last observation) - drop those
final = final.dropna().reset_index()
final["Time"] = final["Time"].dt.total_seconds()  # back to plain seconds, easier to work with downstream

# OMNeT++'s vector names carry a trailing space (e.g. "servingCell "); strip it
final.columns = [c.strip() for c in final.columns]

# cosmetic: keep Time/Object as the first two columns
final = final[["Time", "Object"] + [c for c in final.columns if c not in ("Time", "Object")]]

assert not final.isnull().values.any(), "unexpected NaNs remain after resampling"
final.head()


## 4. Add lagged / lead `servingCell` columns

For each of `LAG_LEAD_SECONDS`, add a column with the serving cell N
seconds in the past (`servingCell-N`) and N seconds in the future
(`servingCellN`). A value of `-1` means "no data available yet" (start or
end of the vehicle's trip) rather than a real cell id.


In [ ]:
# how many resampled rows correspond to one second, given RESAMPLE_FREQ
# (e.g. 100 rows/second at the default 10ms grid)
resample_seconds = pd.Timedelta(RESAMPLE_FREQ).total_seconds()

for sec in LAG_LEAD_SECONDS:
    shift_rows = round(sec / resample_seconds)
    # shift(+N) looks backwards in time -> "what cell was this vehicle on N seconds ago"
    final[f"servingCell-{sec}"] = final.groupby("Object")["servingCell"].shift(shift_rows).fillna(-1)
    # shift(-N) looks forwards in time -> "what cell will this vehicle be on N seconds from now"
    final[f"servingCell{sec}"] = final.groupby("Object")["servingCell"].shift(-shift_rows).fillna(-1)

final.head()


## 5. Save the output matrix

In [ ]:
# tab-separated to stay consistent with the raw OMNeT++ export format used upstream
final.to_csv(OUTPUT_MATRIX_PATH, index=False, sep="\t")
print(f"saved {OUTPUT_MATRIX_PATH} - shape {final.shape}")
